In [116]:
import polars as pl
from path_config import PathConfig
paths = PathConfig()

In [117]:
dl_data = pl.read_excel(paths.dl_path, sheet_name='Series Formatted Data')
dl_test_data = pl.read_excel(paths.test_dl_path, sheet_name='Series Formatted Data')
kabinets = pl.read_csv((paths._processed_saha_olcum_dir / "Kabinets.csv"))

Could not determine dtype for column 10, falling back to string
Could not determine dtype for column 11, falling back to string
Could not determine dtype for column 12, falling back to string
Could not determine dtype for column 13, falling back to string
Could not determine dtype for column 15, falling back to string
Could not determine dtype for column 16, falling back to string
Could not determine dtype for column 17, falling back to string
Could not determine dtype for column 18, falling back to string
Could not determine dtype for column 20, falling back to string
Could not determine dtype for column 21, falling back to string
Could not determine dtype for column 22, falling back to string
Could not determine dtype for column 23, falling back to string
Could not determine dtype for column 24, falling back to string
Could not determine dtype for column 27, falling back to string
Could not determine dtype for column 28, falling back to string
Could not determine dtype for column 29,

In [118]:
def join_main_bs_data(data, bs_data, main_bs_col_name="NR_UE_PCI_0"):
    data = data.join(
        bs_data.select(
            ["PCI", "Latitude", "Longitude", "Azimuth", "Vertical_Beamwidth", "MIMO"]
        ).rename(
            {
                "Latitude": "BS_LATITUDE",
                "Longitude": "BS_LONGITUDE",
                "Azimuth": "BS_AZIMUTH",
                "Vertical_Beamwidth": "BS_VERTICAL_BEAMWIDTH",
            }
        ),
        left_on=main_bs_col_name,
        right_on="PCI",
        how="left",
    )

    data = data.drop(main_bs_col_name)

    return data

In [ ]:
def aggregate_nbr_rsrp(data, n, iterate_n = True ,drop_pci_cols = False):

    nbr_rsrp_cols = [col for col in dl_test_data.columns if col.startswith("NR_UE_Nbr_RSRP_")]
    nbr_pci_cols = [col for col in dl_test_data.columns if col.startswith("NR_UE_Nbr_PCI_")]

    if n>len(nbr_pci_cols):
        n = len(nbr_pci_cols)

    def get_top_pci(pairs_list, n):
        valid_pairs = [p for p in pairs_list if p["rsrp"] is not None and p["pci"] is not None]
        if not valid_pairs:
            return None
            
        return [int(pair["pci"]) for pair in sorted(valid_pairs, key=lambda x: float(x["rsrp"]), reverse=True)[:n]]

    if iterate_n:
        for m in range (1,n+1):
            result = (
                data.with_row_index()
                .select([
                    "index",
                    pl.concat_list([
                        pl.struct([
                            pl.col(nbr_rsrp_cols[i]).alias("rsrp"), 
                            pl.col(nbr_pci_cols[i]).alias("pci")
                            ]) 
                        for i in range(len(nbr_pci_cols))
                        ]).alias("pairs")
                ])
                .with_columns([                   
                    pl.col("pairs")
                    .map_elements(lambda x: get_top_pci(x, m), return_dtype=pl.List(pl.Int64))
                    .alias(f"top_{m}_ids")
                ])
                .explode(f"top_{m}_ids")
                .join(kabinets[["PCI","Longitude", "Latitude"]], left_on=f"top_{m}_ids", right_on="PCI")
                .group_by("index")
                .agg([                                                
                    pl.col("Longitude").mean().alias(f"TOP_{m}_NBR_PCI_LONGITUDE_AVG"),      
                    pl.col("Latitude").mean().alias(f"TOP_{m}_NBR_PCI_LATITUDE_AVG")       
                ]) 
            )

            data = data.with_row_index("index").join(result, how="left", on="index").drop("index")
    else:
        result = (
            data.with_row_index()
            .select([
                "index",
                pl.concat_list([
                    pl.struct([
                        pl.col(nbr_rsrp_cols[i]).alias("rsrp"), 
                        pl.col(nbr_pci_cols[i]).alias("pci")
                        ]) 
                    for i in range(len(nbr_pci_cols))
                    ]).alias("pairs")
            ])
            .with_columns([                   
                pl.col("pairs")
                .map_elements(lambda x: get_top_pci(x, m), return_dtype=pl.List(pl.Int64))
                .alias(f"top_{m}_ids")
            ])
            .explode(f"top_{m}_ids")
            .join(kabinets[["PCI","Longitude", "Latitude"]], left_on=f"top_{m}_ids", right_on="PCI")
            .group_by("index")
            .agg([                                                
                pl.col("Longitude").mean().alias(f"TOP_{m}_NBR_PCI_LONGITUDE_AVG"),      
                pl.col("Latitude").mean().alias(f"TOP_{m}_NBR_PCI_LATITUDE_AVG")       
            ]) 
        )

        data = data.with_row_index("index").join(result, how="left", on="index").drop("index")
    
    if drop_pci_cols:
        data = data.drop(nbr_pci_cols)

    return data

In [120]:
dl_data = join_main_bs_data(dl_data, kabinets)
dl_data = aggregate_nbr_rsrp(dl_data, 4, True)
dl_data

Message,Time,Longitude,Latitude,Technology_Mode,NR_UE_RSRP_0,NR_UE_RSRQ_0,NR_UE_SINR_0,NR_UE_Nbr_PCI_0,NR_UE_Nbr_PCI_1,NR_UE_Nbr_PCI_2,NR_UE_Nbr_PCI_3,NR_UE_Nbr_PCI_4,NR_UE_Nbr_RSRP_0,NR_UE_Nbr_RSRP_1,NR_UE_Nbr_RSRP_2,NR_UE_Nbr_RSRP_3,NR_UE_Nbr_RSRP_4,NR_UE_Nbr_RSRQ_0,NR_UE_Nbr_RSRQ_1,NR_UE_Nbr_RSRQ_2,NR_UE_Nbr_RSRQ_3,NR_UE_Nbr_RSRQ_4,NR_UE_Timing_Advance,NR_UE_Pathloss_DL_0,NR_UE_Throughput_PDCP_DL,App_Throughput_DL,NR_UE_NACK_Rate_DL_0,NR_UE_Ack_As_Nack_DL_0,NR_UE_MCS_DL_0,NR_UE_RB_Num_DL_0,NR_UE_Modulation_Avg_DL_0,NR_UE_RI_DL_0,NR_UE_BLER_DL_0,NR_UE_CCE_AggregationLev_0,NR_UE_Power_Tx_PUSCH_0,NR_UE_Power_Tx_PRACH_0,NR_UE_NACK_Rate_UL_0,NR_UE_RACH_Attempt,NR_UE_RACH_OK,NR_UE_RACH_Fail,NR_UE_RACH_Procedure_Count,NR_UE_RRCReEstAttempt,NR_UE_RRCReEstFail,NR_UE_RRCReEst_EndResult,NR_UE_RRCConnectionAttempt,NR_UE_RRCConnectionSetupOk,NR_UE_RRCConnectionComplete,NR_UE_RRCConnectionDrop,NR_UE_RRCHOAttempt,NR_UE_RRCHOOK,NR_RRC_MsgType,NAS_5GS_MM_MessageType,NAS_5GS_SM_MessageType,BS_LATITUDE,BS_LONGITUDE,BS_AZIMUTH,BS_VERTICAL_BEAMWIDTH,MIMO,TOP_1_NBR_PCI_LONGITUDE,TOP_1_NBR_PCI_LATITUDE,TOP_2_NBR_PCI_LONGITUDE,TOP_2_NBR_PCI_LATITUDE,TOP_3_NBR_PCI_LONGITUDE,TOP_3_NBR_PCI_LATITUDE,TOP_4_NBR_PCI_LONGITUDE,TOP_4_NBR_PCI_LATITUDE
i64,datetime[ms],f64,f64,str,f64,f64,f64,i64,str,str,str,str,f64,str,str,str,str,f64,str,str,str,str,str,f64,f64,str,str,str,str,str,str,str,str,str,f64,str,f64,i64,i64,i64,str,i64,i64,str,i64,i64,i64,i64,i64,i64,str,str,str,f64,f64,i64,f64,str,f64,f64,f64,f64,f64,f64,f64,f64
0,2025-03-14 12:14:33.127,null,null,"""5GSA""",null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,0,0,0,null,0,0,null,0,0,0,0,0,0,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null
1,2025-03-14 12:14:33.444,null,null,"""5GSA""",-87.1,-11.2,7.3,76,null,null,null,null,-97.3,null,null,null,null,-16.3,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,0,0,0,null,0,0,null,0,0,0,0,0,0,null,null,null,41.108086,29.028122,110,10.0,"""64T64R""",29.027833,41.105469,29.027833,41.105469,29.027833,41.105469,29.027833,41.105469
2,2025-03-14 12:14:33.444,29.02949,41.10723,"""5GSA""",null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,0,0,0,null,0,0,null,0,0,0,0,0,0,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null
3,2025-03-14 12:14:33.444,29.02949,41.10723,"""5GSA""",null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,0,0,0,null,0,0,null,0,0,0,0,0,0,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null
4,2025-03-14 12:14:33.444,29.02949,41.10723,"""5GSA""",null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,0,0,0,null,0,0,null,0,0,0,0,0,0,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
72475,2025-03-14 12:32:17.650,29.02143,41.10031,"""5GSA""",null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,0,0,0,null,0,0,null,0,0,0,0,0,0,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null
72476,2025-03-14 12:32:17.781,29.02143,41.10031,"""5GSA""",null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,0,0,0,null,0,0,null,0,0,0,0,0,0,null,null,null,null,null,null,null,null,null,null,null